# Fraud Detection Baseline Models

本 notebook 針對 `Transactions Data.csv` 建立 baseline：

1. Logistic Regression  
2. Decision Tree  
3. Random Forest（可選，較耗時）

重點：
- 以 `step` 做 **time-based split**（避免資料洩漏）
- 指標以 **PR-AUC / Recall / Precision / F1** 為主
- 產出 **risk tier（Low/Medium/High）** 做風險分級


In [ ]:
# ==== 0) Imports ====
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)


In [ ]:
# ==== 1) Config ====
DATA_PATH = 'Transactions Data.csv'

# 若你想先快速跑流程，可啟用抽樣（例如 0.2 代表抽 20%）
USE_SAMPLE = False
SAMPLE_FRAC = 0.2
RANDOM_STATE = 42

# time split 比例
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO  = 0.15

# 是否訓練 RandomForest（資料大時可能較久）
RUN_RANDOM_FOREST = True


In [ ]:
# ==== 2) Load data ====
df = pd.read_csv(DATA_PATH)

if USE_SAMPLE:
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE).sort_values('step').reset_index(drop=True)

print('Shape:', df.shape)
print(df.head(3))
print('
Fraud rate:', df['isFraud'].mean())


In [ ]:
# ==== 3) Feature engineering ====
work = df.copy()

# delta features
work['deltaOrig'] = work['oldbalanceOrg'] - work['newbalanceOrig']
work['deltaDest'] = work['newbalanceDest'] - work['oldbalanceDest']

# zero-balance indicators
work['isOrigBalanceZero'] = (work['oldbalanceOrg'] == 0).astype(int)
work['isDestBalanceZero'] = (work['oldbalanceDest'] == 0).astype(int)

# 先不使用高基數 ID，避免 baseline 過擬合 / 記憶化
work = work.drop(columns=['nameOrig', 'nameDest'])

TARGET = 'isFraud'
features = [c for c in work.columns if c != TARGET]

num_features = [
    'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'deltaOrig', 'deltaDest',
    'isOrigBalanceZero', 'isDestBalanceZero', 'isFlaggedFraud'
]
cat_features = ['type']

X = work[features]
y = work[TARGET]

print('Num features:', num_features)
print('Cat features:', cat_features)


In [ ]:
# ==== 4) Time-based split by step ====
steps_sorted = np.sort(work['step'].unique())

train_cut_idx = int(len(steps_sorted) * TRAIN_RATIO)
valid_cut_idx = int(len(steps_sorted) * (TRAIN_RATIO + VALID_RATIO))

train_max_step = steps_sorted[train_cut_idx - 1]
valid_max_step = steps_sorted[valid_cut_idx - 1]

train_mask = work['step'] <= train_max_step
valid_mask = (work['step'] > train_max_step) & (work['step'] <= valid_max_step)
test_mask  = work['step'] > valid_max_step

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_valid, y_valid = X.loc[valid_mask], y.loc[valid_mask]
X_test,  y_test  = X.loc[test_mask],  y.loc[test_mask]

print('Train:', X_train.shape, 'Fraud rate:', y_train.mean())
print('Valid:', X_valid.shape, 'Fraud rate:', y_valid.mean())
print('Test :', X_test.shape,  'Fraud rate:', y_test.mean())
print('Step range train <=', train_max_step, '| valid <=', valid_max_step)


In [ ]:
# ==== 5) Preprocessors ====
preprocess_lr = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features),
    ],
    remainder='drop'
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features),
    ],
    remainder='drop'
)


In [ ]:
# ==== 6) Evaluation helpers ====
def evaluate_binary(y_true, y_prob, threshold=0.5, name='model'):
    y_pred = (y_prob >= threshold).astype(int)
    ap = average_precision_score(y_true, y_prob)
    roc = roc_auc_score(y_true, y_prob)
    cm = confusion_matrix(y_true, y_pred)

    print(f'[{name}] threshold={threshold:.3f}')
    print(f'PR-AUC (Average Precision): {ap:.6f}')
    print(f'ROC-AUC: {roc:.6f}')
    print('Confusion matrix:
', cm)
    print(classification_report(y_true, y_pred, digits=4))

    return {'model': name, 'threshold': threshold, 'pr_auc': ap, 'roc_auc': roc}


def threshold_table(y_true, y_prob, thresholds=(0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5)):
    rows = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append((t, precision, recall, f1, int((y_pred==1).sum())))

    return pd.DataFrame(rows, columns=['threshold','precision','recall','f1','predicted_positive'])


def plot_pr_curve(y_true, y_prob, title='PR Curve'):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    plt.figure(figsize=(6,4))
    plt.plot(r, p, label=f'AP={ap:.4f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


def assign_risk_tier(prob, t_mid=0.10, t_high=0.30):
    return pd.cut(
        prob,
        bins=[-np.inf, t_mid, t_high, np.inf],
        labels=['Low', 'Medium', 'High']
    )


In [ ]:
# ==== 7) Logistic Regression baseline ====
logit = Pipeline([
    ('prep', preprocess_lr),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        solver='saga',
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

logit.fit(X_train, y_train)

valid_prob_logit = logit.predict_proba(X_valid)[:, 1]
test_prob_logit  = logit.predict_proba(X_test)[:, 1]

plot_pr_curve(y_valid, valid_prob_logit, title='Logistic Regression - Valid PR Curve')

logit_valid_metrics = evaluate_binary(y_valid, valid_prob_logit, threshold=0.5, name='Logistic (Valid)')
print('
Threshold scan (Valid):')
display(threshold_table(y_valid, valid_prob_logit))


In [ ]:
# ==== 8) Decision Tree baseline ====
dtree = Pipeline([
    ('prep', preprocess_tree),
    ('clf', DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=200,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

dtree.fit(X_train, y_train)

valid_prob_tree = dtree.predict_proba(X_valid)[:, 1]
test_prob_tree  = dtree.predict_proba(X_test)[:, 1]

plot_pr_curve(y_valid, valid_prob_tree, title='Decision Tree - Valid PR Curve')

tree_valid_metrics = evaluate_binary(y_valid, valid_prob_tree, threshold=0.5, name='DecisionTree (Valid)')
print('
Threshold scan (Valid):')
display(threshold_table(y_valid, valid_prob_tree))


In [ ]:
# ==== 9) Random Forest baseline (optional) ====
rf_valid_metrics = None
if RUN_RANDOM_FOREST:
    rf = Pipeline([
        ('prep', preprocess_tree),
        ('clf', RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            min_samples_leaf=100,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])

    rf.fit(X_train, y_train)

    valid_prob_rf = rf.predict_proba(X_valid)[:, 1]
    test_prob_rf  = rf.predict_proba(X_test)[:, 1]

    plot_pr_curve(y_valid, valid_prob_rf, title='Random Forest - Valid PR Curve')

    rf_valid_metrics = evaluate_binary(y_valid, valid_prob_rf, threshold=0.5, name='RandomForest (Valid)')
    print('
Threshold scan (Valid):')
    display(threshold_table(y_valid, valid_prob_rf))


In [ ]:
# ==== 10) Compare models on validation PR-AUC ====
rows = [logit_valid_metrics, tree_valid_metrics]
if rf_valid_metrics is not None:
    rows.append(rf_valid_metrics)

valid_compare = pd.DataFrame(rows).sort_values('pr_auc', ascending=False)
display(valid_compare)


In [ ]:
# ==== 11) Pick model + threshold, then evaluate on TEST ====
# 你可以改成 valid 表現最佳的模型
SELECT_MODEL = 'logit'  # 'logit' | 'tree' | 'rf'
SELECT_THRESHOLD = 0.10 # Recall-first 常會用較低 threshold

if SELECT_MODEL == 'logit':
    test_prob = test_prob_logit
    model_name = 'Logistic'
elif SELECT_MODEL == 'tree':
    test_prob = test_prob_tree
    model_name = 'DecisionTree'
else:
    if not RUN_RANDOM_FOREST:
        raise ValueError('RUN_RANDOM_FOREST=False，無法選 rf')
    test_prob = test_prob_rf
    model_name = 'RandomForest'

evaluate_binary(y_test, test_prob, threshold=SELECT_THRESHOLD, name=f'{model_name} (Test)')
print('
Threshold scan (Test):')
display(threshold_table(y_test, test_prob))


In [ ]:
# ==== 12) Risk tier report (Low / Medium / High) ====
# 建議先用 Valid 決定 t_mid / t_high，再固定到 Test
T_MID = 0.10
T_HIGH = 0.30

risk_df = pd.DataFrame({
    'y_true': y_test.values,
    'prob': test_prob
})
risk_df['risk_tier'] = assign_risk_tier(risk_df['prob'], t_mid=T_MID, t_high=T_HIGH)

summary = risk_df.groupby('risk_tier', observed=True).agg(
    volume=('y_true', 'size'),
    fraud_rate=('y_true', 'mean'),
    fraud_count=('y_true', 'sum')
).reset_index()

summary['volume_pct'] = summary['volume'] / summary['volume'].sum()
summary = summary.sort_values('risk_tier')
display(summary)


## 下一步建議

1. 在 Valid 集上以業務條件決定 `t_mid` / `t_high`（例如人工審核量上限）。
2. 加入時間窗口特徵（帳戶近期交易次數、累積金額、異常波動）。
3. 再比較更強模型（例如 LightGBM / XGBoost）是否在同等 Precision 下提高 Recall。
